# 2.1 Augmentation Pipeline

In [1]:
from pathlib import Path
import cv2
import numpy as np
import albumentations as A

# Source output from 1_roi_pipeline.
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = Path.cwd().parent

data_root = project_root / "data"


def require_dir(root: Path, folder_name: str, guidance: str) -> Path:
    path = root / folder_name
    if not path.exists() or not path.is_dir():
        raise FileNotFoundError(
            f"Required folder not found: {path}. {guidance}"
        )
    return path


SOURCE_TRAIN_DIR = require_dir(
    data_root,
    "6_enhanced_roi_train",
    "Run processing/1_roi_pipeline.ipynb first to generate required folders.",
)
SOURCE_TEST_DIR = require_dir(
    data_root,
    "6_enhanced_roi_test",
    "Run processing/1_roi_pipeline.ipynb first to generate required folders.",
)
FINAL_PROCESSED_DIR = data_root / "7_final_processed"
AUGMENTED_DIR = data_root / "7_augmented"
TEST_PROCESSED_DIR = data_root / "7_test_processed"

OUTPUT_SIZE = (128, 128)
TARGET_PER_PERSON = 100
IMAGE_QUALITY = 95
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for folder in [FINAL_PROCESSED_DIR, AUGMENTED_DIR, TEST_PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Augmentation pipeline configured")
print(f"  Source train: {SOURCE_TRAIN_DIR}")
print(f"  Source test : {SOURCE_TEST_DIR}")
print(f"  Final train : {FINAL_PROCESSED_DIR}")
print(f"  Augmented   : {AUGMENTED_DIR}")
print(f"  Test proc   : {TEST_PROCESSED_DIR}")
print(f"  Output size : {OUTPUT_SIZE}")
print(f"  Target/person: {TARGET_PER_PERSON}")


class ROIDataAugmentor:
    def __init__(self, target_per_person: int = TARGET_PER_PERSON):
        self.target_per_person = target_per_person
        self.pipelines = {
            "illumination": A.Compose([
                A.OneOf([
                    A.RandomBrightnessContrast(brightness_limit=0.30, contrast_limit=0.30, p=1.0),
                    A.RandomGamma(gamma_limit=(50, 150), p=1.0),
                    A.CLAHE(clip_limit=4.0, tile_grid_size=(4, 4), p=1.0),
                ], p=1.0),
            ]),
            "noise": A.Compose([
                A.OneOf([
                    A.GaussNoise(std_range=(0.02, 0.08), mean_range=(0.0, 0.0), p=1.0),
                    A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.10, 0.30), p=1.0),
                ], p=1.0),
            ]),
            "geometric": A.Compose([
                A.OneOf([
                    A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
                    A.HorizontalFlip(p=1.0),
                    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.03, 0.03), p=1.0),
                ], p=1.0),
            ]),
            "combined": A.Compose([
                A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
                A.OneOf([
                    A.GaussNoise(std_range=(0.02, 0.08), mean_range=(0.0, 0.0), p=1.0),
                    A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.10, 0.30), p=1.0),
                ], p=0.5),
                A.OneOf([
                    A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
                    A.HorizontalFlip(p=1.0),
                    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.03, 0.03), p=1.0),
                ], p=0.5),
                A.RandomGamma(gamma_limit=(60, 140), p=0.3),
            ]),
            "diverse": A.Compose([
                A.OneOf([
                    A.RandomBrightnessContrast(brightness_limit=0.30, contrast_limit=0.30, p=1.0),
                    A.RandomGamma(gamma_limit=(50, 150), p=1.0),
                    A.CLAHE(clip_limit=4.0, tile_grid_size=(4, 4), p=1.0),
                ], p=0.7),
                A.OneOf([
                    A.Rotate(limit=20, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
                    A.Affine(scale=(0.90, 1.10), translate_percent=(-0.06, 0.06), rotate=(-20, 20), shear=(-8, 8), p=1.0),
                    A.Perspective(scale=(0.03, 0.08), keep_size=True, p=1.0),
                ], p=0.7),
                A.OneOf([
                    A.GaussNoise(std_range=(0.02, 0.10), mean_range=(0.0, 0.0), p=1.0),
                    A.ISONoise(color_shift=(0.01, 0.04), intensity=(0.10, 0.35), p=1.0),
                    A.GridDistortion(num_steps=5, distort_limit=0.1, p=1.0),
                ], p=0.5),
            ]),
        }

    def _resize_gray(self, image_gray: np.ndarray) -> np.ndarray:
        if image_gray is None:
            return None
        if image_gray.shape[:2] != OUTPUT_SIZE:
            image_gray = cv2.resize(image_gray, OUTPUT_SIZE, interpolation=cv2.INTER_CUBIC)
        return image_gray

    def _save_gray(self, file_path: Path, image_gray: np.ndarray):
        file_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(file_path), image_gray, [int(cv2.IMWRITE_JPEG_QUALITY), IMAGE_QUALITY])

    def augment_gray(self, image_gray: np.ndarray, pipeline_name: str = "combined") -> np.ndarray:
        if image_gray is None:
            return None
        pipeline = self.pipelines.get(pipeline_name, self.pipelines["combined"])
        image_bgr = cv2.cvtColor(image_gray, cv2.COLOR_GRAY2BGR)
        augmented = pipeline(image=image_bgr)["image"]
        return cv2.cvtColor(augmented, cv2.COLOR_BGR2GRAY)

    def copy_folder(self, source_folder: Path, output_folder: Path):
        output_folder.mkdir(parents=True, exist_ok=True)
        copied = 0
        unreadable = 0

        for class_dir in sorted([p for p in source_folder.iterdir() if p.is_dir()]):
            input_files = sorted([
                p for p in class_dir.iterdir()
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS
            ])
            if not input_files:
                continue

            output_class_dir = output_folder / class_dir.name
            output_class_dir.mkdir(parents=True, exist_ok=True)
            for old_file in output_class_dir.glob("*.jpg"):
                old_file.unlink(missing_ok=True)

            for file_path in input_files:
                image_gray = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
                if image_gray is None:
                    unreadable += 1
                    continue
                resized = self._resize_gray(image_gray)
                self._save_gray(output_class_dir / file_path.name, resized)
                copied += 1

        return copied, unreadable

    def augment_person(self, person_name: str, input_folder: Path, output_folder: Path):
        person_input = input_folder / person_name
        if not person_input.exists():
            return 0, 0, 0

        image_files = sorted([
            p for p in person_input.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        ])
        if not image_files:
            return 0, 0, 0

        loaded = {}
        for file_path in image_files:
            image_gray = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
            if image_gray is not None:
                loaded[file_path.name] = self._resize_gray(image_gray)

        valid_files = list(loaded.keys())
        if not valid_files:
            return len(image_files), 0, 0

        output_person = output_folder / person_name
        output_person.mkdir(parents=True, exist_ok=True)
        for old_file in output_person.glob("*.jpg"):
            old_file.unlink(missing_ok=True)

        # Copy originals first.
        for file_name, image_gray in loaded.items():
            self._save_gray(output_person / file_name, image_gray)

        originals = len(valid_files)
        needed = max(0, self.target_per_person - originals)
        pipeline_names = list(self.pipelines.keys())
        generated = 0
        cycle_index = 0

        while generated < needed:
            for file_name in valid_files:
                if generated >= needed:
                    break
                image_gray = loaded[file_name]
                pipeline_name = pipeline_names[cycle_index % len(pipeline_names)]
                try:
                    augmented_gray = self.augment_gray(image_gray, pipeline_name=pipeline_name)
                    augmented_gray = self._resize_gray(augmented_gray)
                    base_name = Path(file_name).stem
                    augmented_name = f"{base_name}_aug_{pipeline_name}_{cycle_index:03d}.jpg"
                    self._save_gray(output_person / augmented_name, augmented_gray)
                    generated += 1
                except Exception as error:
                    print(f"Skipping augmentation for {person_name}/{file_name} using {pipeline_name}: {error}")
                cycle_index += 1

        return originals, generated, originals + generated


augmentor = ROIDataAugmentor()
print("ROIDataAugmentor ready")

Augmentation pipeline configured
  Source train: c:\Users\harry\Documents\school\ip\AttSystem\data\6_enhanced_roi_train
  Source test : c:\Users\harry\Documents\school\ip\AttSystem\data\6_enhanced_roi_test
  Final train : c:\Users\harry\Documents\school\ip\AttSystem\data\7_final_processed
  Augmented   : c:\Users\harry\Documents\school\ip\AttSystem\data\7_augmented
  Test proc   : c:\Users\harry\Documents\school\ip\AttSystem\data\7_test_processed
  Output size : (128, 128)
  Target/person: 100
ROIDataAugmentor ready


In [2]:
def build_training_inputs():
    if not SOURCE_TRAIN_DIR.exists() or not SOURCE_TEST_DIR.exists():
        print("Missing data/6_enhanced_roi_train or data/6_enhanced_roi_test. Run 1_roi_pipeline first.")
        return None

    print("Building training inputs from ROI outputs")
    print("=" * 60)

    # Baseline copies for the non-augmented training path and test evaluation path.
    copied_train, unreadable_train = augmentor.copy_folder(SOURCE_TRAIN_DIR, FINAL_PROCESSED_DIR)
    copied_test, unreadable_test = augmentor.copy_folder(SOURCE_TEST_DIR, TEST_PROCESSED_DIR)

    print(f"Baseline train copies : {copied_train}")
    print(f"Baseline test copies  : {copied_test}")
    print(f"Unreadable train files : {unreadable_train}")
    print(f"Unreadable test files  : {unreadable_test}")

    person_names = sorted([p.name for p in SOURCE_TRAIN_DIR.iterdir() if p.is_dir()])
    if not person_names:
        print("No person folders found in the training source.")
        return None

    augmentation_rows = []
    total_originals = 0
    total_augmented = 0

    for person_name in person_names:
        originals, generated, final_total = augmentor.augment_person(person_name, FINAL_PROCESSED_DIR, AUGMENTED_DIR)
        if originals == 0:
            continue
        augmentation_rows.append((person_name, originals, generated, final_total))
        total_originals += originals
        total_augmented += generated

    print("\nAugmentation complete")
    print(f"  People processed : {len(augmentation_rows)}")
    print(f"  Originals copied : {total_originals}")
    print(f"  Augmented added  : {total_augmented}")
    print(f"  Final train set   : {total_originals + total_augmented}")
    print(f"  Train output      : {FINAL_PROCESSED_DIR}")
    print(f"  Augmented output  : {AUGMENTED_DIR}")
    print(f"  Test output       : {TEST_PROCESSED_DIR}")

    return augmentation_rows


augmentation_results = build_training_inputs()

Building training inputs from ROI outputs
Baseline train copies : 299
Baseline test copies  : 83
Unreadable train files : 0
Unreadable test files  : 0

Augmentation complete
  People processed : 21
  Originals copied : 299
  Augmented added  : 1801
  Final train set   : 2100
  Train output      : c:\Users\harry\Documents\school\ip\AttSystem\data\7_final_processed
  Augmented output  : c:\Users\harry\Documents\school\ip\AttSystem\data\7_augmented
  Test output       : c:\Users\harry\Documents\school\ip\AttSystem\data\7_test_processed


# 2.2 Dataset Analysis

In [3]:
def count_person_images(folder: Path, person_name: str) -> int:
    if not folder.exists():
        return 0
    person_dir = folder / person_name
    if not person_dir.exists() or not person_dir.is_dir():
        return 0
    return sum(1 for file_path in person_dir.iterdir() if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTS)


def get_people() -> list[str]:
    if SOURCE_TRAIN_DIR.exists():
        return sorted([p.name for p in SOURCE_TRAIN_DIR.iterdir() if p.is_dir()])
    if SOURCE_TEST_DIR.exists():
        return sorted([p.name for p in SOURCE_TEST_DIR.iterdir() if p.is_dir()])
    return []


people = get_people()
if not people:
    print("No person folders found in the ROI source folders.")
else:
    columns = ["Person", "Raw", "ROI Train", "ROI Test", "7 Final", "7 Aug", "7 Test", "Retention %"]
    rows = []

    for person_name in people:
        raw_count = count_person_images(data_root / "1_raw", person_name)
        roi_train_count = count_person_images(SOURCE_TRAIN_DIR, person_name)
        roi_test_count = count_person_images(SOURCE_TEST_DIR, person_name)
        final_proc_count = count_person_images(FINAL_PROCESSED_DIR, person_name)
        augmented_count = count_person_images(AUGMENTED_DIR, person_name)
        test_proc_count = count_person_images(TEST_PROCESSED_DIR, person_name)
        final_total = final_proc_count + augmented_count
        retention = (final_total / raw_count * 100.0) if raw_count else 0.0

        rows.append([
            person_name,
            raw_count,
            roi_train_count,
            roi_test_count,
            final_proc_count,
            augmented_count,
            test_proc_count,
            f"{retention:.1f}%",
        ])

    widths = [len(col) for col in columns]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(str(value)))

    def format_row(values):
        return " | ".join(str(value).ljust(widths[index]) for index, value in enumerate(values))

    print(format_row(columns))
    print("-|-".join("-" * width for width in widths))
    for row in rows:
        print(format_row(row))

    print("=" * 110)
    total_raw = sum(int(row[1]) for row in rows)
    total_final = sum(int(row[4]) + int(row[5]) for row in rows)
    overall_retention = (total_final / total_raw * 100.0) if total_raw else 0.0
    print(f"Total raw images      : {total_raw}")
    print(f"Total final train imgs : {total_final}")
    print(f"Overall retention     : {overall_retention:.1f}%")
    print("Dataset analysis complete.")

Person      | Raw | ROI Train | ROI Test | 7 Final | 7 Aug | 7 Test | Retention %
------------|-----|-----------|----------|---------|-------|--------|------------
benjamin    | 11  | 13        | 4        | 13      | 100   | 4      | 1027.3%    
chern_tak   | 0   | 22        | 6        | 22      | 100   | 6      | 0.0%       
chillien    | 9   | 6         | 2        | 6       | 100   | 2      | 1177.8%    
daniel      | 0   | 15        | 4        | 15      | 100   | 4      | 0.0%       
dylan       | 0   | 12        | 1        | 12      | 100   | 1      | 0.0%       
han_soon    | 12  | 15        | 4        | 15      | 100   | 4      | 958.3%     
harry       | 10  | 7         | 2        | 7       | 100   | 2      | 1070.0%    
isaac       | 0   | 6         | 2        | 6       | 100   | 2      | 0.0%       
jing_ang    | 21  | 20        | 5        | 20      | 100   | 5      | 571.4%     
jun_wei     | 0   | 2         | 0        | 2       | 100   | 0      | 0.0%       
kang_kai    | 10